# 05 · Schere–Stein–Papier

Führe die Codezellen mit **Shift + Enter** aus. Lies jeweils zuerst die Frage, probiere dann das Werkzeug aus und besprecht kurz eure Beobachtung. Notizen kannst du direkt bei deinen Papieraufgaben machen. Speichere deine Codeänderungen mit **Strg + S**.


**Schere schlägt Papier, Stein schlägt Schere, Papier schlägt Stein.** Der Computer kontert seine Prognose deines nächsten Zugs.

**Schwierigkeitsstufen**

| Stufe | Orientierung |
|---|---|
| Leicht | Ab Klasse 9: Wahrscheinlichkeiten, Baumdiagramme, Bruchrechnen. Oft genügen Ausprobieren und genaues Lesen. |
| Schwerer | Ab Klasse 10/11: Matrizen, Gleichungssysteme, mehrstufige Überlegungen und Argumentieren mit Begriffen aus dem Skript. |
| Schwer | Eher Klasse 12/13: Knobelaufgaben, anspruchsvollere Begründungen, Beweisideen und ungewohnte Modellierung. |

Die Klassenstufen sind eine Orientierung, keine Zugangsvoraussetzung. Du kannst schwierigere Teilaufgaben auch überspringen, die Reihenfolge ist nicht obligatorisch.


<br>

## Vorbereitung der Widgets
#### Klicke zuerst oben im Menü auf **Run → Run All Cells**. Den Code in der hier folgenden Zelle brauchst du nicht zu bearbeiten.

In [1]:
# Pakete und Widgets vorbereiten.
import sys

try:
    import ipywidgets as W
except ModuleNotFoundError:
    if sys.platform == "emscripten":
        import piplite

        await piplite.install("ipywidgets>=8,<9")
        import ipywidgets as W
    else:
        raise RuntimeError("Bitte ipywidgets in dieser Python-Umgebung installieren lassen.")


import io
import html
import random
from collections import Counter, defaultdict, deque
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ZUEGE = ("Schere", "Stein", "Papier")
GEGNER = ("Dreierzyklus", "Viererfolge", "Häufig wiederholen", "Gleichverteilung")
PANELS = {}


# Spielregeln, Lernmodell und gemeinsame Widget-Hilfen.
def konter(zug):
    return (zug + 1) % 3


def ergebnis(computer, gegner):
    return 0 if computer == gegner else (1 if (computer - gegner) % 3 == 1 else -1)


class MarkovSpiel:
    def __init__(self, ordnung=2, seed=2026, fenster=None):
        if ordnung not in (1, 2, 3):
            raise ValueError("Ordnung muss 1, 2 oder 3 sein.")
        if fenster is not None and fenster < 1:
            raise ValueError("Das Fenster muss positiv sein.")
        self.ordnung = ordnung
        self.rng = random.Random(seed)
        self.fenster = fenster
        self.historie = []
        self.tabelle = defaultdict(Counter)
        self.ereignisse = deque()
        self.prepare()

    def prepare(self):
        self.kontext = (
            tuple(self.historie[-self.ordnung :])
            if len(self.historie) >= self.ordnung
            else None
        )
        self.vorher = dict(self.tabelle.get(self.kontext, {}))
        maximal = max(self.vorher.values(), default=0)
        kandidaten = [i for i in range(3) if self.vorher.get(i, 0) == maximal]
        self.prognose = self.rng.choice(kandidaten)
        self.computer = konter(self.prognose)  # Steht VOR der Eingabe der nächsten Runde fest.

    def runde(self, gegner):
        if gegner not in range(3):
            raise ValueError("Ungültiger Zug.")
        record = {
            "computer": self.computer,
            "gegner": gegner,
            "prognose": self.prognose,
            "kontext": self.kontext,
            "counts_before": self.vorher.copy(),
            "score": ergebnis(self.computer, gegner),
        }
        if self.kontext is not None:
            self.tabelle[self.kontext][gegner] += 1
            if self.fenster is not None:
                self.ereignisse.append((self.kontext, gegner))
                if len(self.ereignisse) > self.fenster:
                    k, g = self.ereignisse.popleft()
                    self.tabelle[k][g] -= 1
                    if self.tabelle[k][g] == 0:
                        del self.tabelle[k][g]
                    if not self.tabelle[k]:
                        del self.tabelle[k]
        self.historie.append(gegner)
        self.prepare()
        return record


def gegnerfolge(art, n, seed=2026, wiederholen=0.8):
    rng = random.Random(seed + 10000)
    x = np.empty(n, dtype=int)
    for t in range(n):
        if art == "Dreierzyklus":
            x[t] = t % 3
        elif art == "Viererfolge":
            x[t] = (0, 0, 1, 2)[t % 4]
        elif art == "Gleichverteilung":
            x[t] = rng.randrange(3)
        elif art == "Häufig wiederholen":
            x[t] = (
                rng.randrange(3)
                if t == 0
                else (
                    x[t - 1]
                    if rng.random() < wiederholen
                    else rng.choice([i for i in range(3) if i != x[t - 1]])
                )
            )
        else:
            raise ValueError("Unbekannter Gegner.")
    return x


def simuliere(x, ordnung=2, seed=2026, fenster=None):
    bot = MarkovSpiel(ordnung, seed, fenster)
    scores = []
    contexts = []
    for g in x:
        scores.append(bot.runde(int(g))["score"])
        contexts.append(len(bot.tabelle))
    return {"scores": np.array(scores), "contexts": np.array(contexts), "bot": bot}


def zufalls_computer(x, seed):
    rng = random.Random(seed + 20000)
    return np.array([ergebnis(rng.randrange(3), int(g)) for g in x])


def feld(text, w):
    return W.VBox([W.HTML(html.escape(text)), w])


def regler(value, maximum=100, minimum=0, step=1):
    return W.IntSlider(
        value=value,
        min=minimum,
        max=maximum,
        step=step,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def knopf(text="Neu auslosen"):
    return W.Button(description=text, button_style="info", layout=W.Layout(width="200px"))


def bild():
    return W.Image(format="png", layout=W.Layout(width="700px", max_width="100%"))


def zeichnen(fig, w):
    with io.BytesIO() as b:
        fig.savefig(b, format="png", dpi=105)
        w.value = b.getvalue()


def tabelle(head, rows):
    h = "".join(
        f'<th style="padding:4px 10px;text-align:left">{html.escape(str(x))}</th>' for x in head
    )
    b = "".join(
        "<tr>"
        + "".join(f'<td style="padding:4px 10px">{html.escape(str(x))}</td>' for x in row)
        + "</tr>"
        for row in rows
    )
    return (
        '<div style="max-height:260px;overflow:auto"><table><tr>'
        + h
        + "</tr>"
        + b
        + "</table></div>"
    )


def kontextname(k):
    return " → ".join(ZUEGE[i] for i in k)


def siegkurve(scores):
    return np.cumsum(scores == 1) / np.arange(1, len(scores) + 1)


# Stationen.
def station_vorhersage():
    ws = [
        W.Text(placeholder="Deine Wahrscheinlichkeit", layout=W.Layout(width="230px"))
        for _ in range(3)
    ]
    options = [("Bitte wählen", None)] + [(s, i) for i, s in enumerate(ZUEGE)]
    antwort = W.Dropdown(options=options, value=None)
    antwort1 = W.Dropdown(options=options, value=None)
    btn = knopf("Prüfen")
    info = W.HTML()
    r = {
        "fields": ws,
        "antwort": antwort,
        "antwort1": antwort1,
        "button": btn,
        "info": info,
        "last": None,
    }

    def reset(_):
        r["last"] = None
        info.value = "Eingabe geändert; erneut prüfen."

    def check(_):
        r["last"] = None
        try:
            p = [float(Fraction(w.value.strip().replace(",", "."))) for w in ws]
        except (ValueError, ZeroDivisionError, OverflowError):
            info.value = "Bitte alle Wahrscheinlichkeiten als Zahl oder Bruch eingeben."
            return
        if not all(0 <= v <= 1 for v in p) or abs(sum(p) - 1) > 1e-6:
            info.value = "Die Wahrscheinlichkeiten müssen zwischen 0 und 1 liegen und sich zu 1 summieren."
            return
        if antwort.value is None or antwort1.value is None:
            info.value = "Wähle für beide Ordnungen noch den Computerzug."
            return
        ok = (
            np.allclose(p, [1 / 3, 0, 2 / 3], atol=1e-6, rtol=0)
            and antwort.value == 0
            and antwort1.value == 2
        )
        info.value = (
            "<b>Richtig.</b> Nach Schere → Stein folgen einmal Schere und zweimal Papier: Der Konter ist Schere. Nach Stein allein folgen zweimal Schere, dreimal Stein und zweimal Papier: Der Konter ist Papier."
            if ok
            else "Zähle für Ordnung 2 nur Fortsetzungen von Schere → Stein, für Ordnung 1 alle Fortsetzungen von Stein. Kontere jeweils den häufigsten Folgezug."
        )
        r["last"] = {"correct": bool(ok)}

    for w in ws + [antwort, antwort1]:
        w.observe(reset, names="value")
    btn.on_click(check)
    display(
        W.VBox(
            [feld(f"P({name} als Nächstes | Schere, Stein)", w) for name, w in zip(ZUEGE, ws)]
            + [
                feld("Computerzug bei Ordnung 2 (Schere, Stein)", antwort),
                feld("Computerzug bei Ordnung 1 (Stein)", antwort1),
                btn,
                info,
            ]
        )
    )
    PANELS["vorhersage"] = r
    return r


def station_spielen():
    order = W.Dropdown(options=[1, 2, 3], value=2, description="Ordnung:")
    buttons = [W.Button(description=s, layout=W.Layout(width="110px")) for s in ZUEGE]
    reset = knopf("Neues Spiel")
    score, last, detail = W.HTML(), W.HTML(), W.HTML()
    box = W.Accordion(children=[detail], selected_index=None)
    box.set_title(0, "Prognose der abgeschlossenen Runde")
    r = {
        "order": order,
        "buttons": buttons,
        "reset": reset,
        "seed": 2026,
        "last": None,
        "records": [],
    }

    def new(_=None):
        r["seed"] += 1
        r["bot"] = MarkovSpiel(order.value, r["seed"])
        r["records"] = []
        r["last"] = None
        score.value = "Du: 0 Siege · Computer: 0 Siege · Unentschieden: 0"
        last.value = "Der Computerzug für Runde 1 steht fest. Wähle deinen Zug."
        detail.value = "Noch keine abgeschlossene Runde."

    def play(g):
        rec = r["bot"].runde(g)
        r["records"].append(rec)
        r["last"] = rec
        values = [a["score"] for a in r["records"]]
        score.value = f"<b>Du: {values.count(-1)} Siege · Computer: {values.count(1)} Siege · Unentschieden: {values.count(0)}</b>"
        outcome = {1: "Computer gewinnt", -1: "Du gewinnst", 0: "Unentschieden"}[rec["score"]]
        last.value = f'Runde {len(values)}: Du {ZUEGE[g]} · Computer {ZUEGE[rec["computer"]]} · {outcome}.<br>Der Computerzug für die nächste Runde steht bereits fest.'
        counts = rec["counts_before"]
        total = sum(counts.values())
        detail.value = (
            "Kontext vor dieser Runde: "
            + (
                kontextname(rec["kontext"])
                if rec["kontext"] is not None
                else "noch zu wenige Züge"
            )
            + ".<br>"
        )
        detail.value += tabelle(
            ["Prognosegrundlage", "Anzahl vorher"],
            [[ZUEGE[i], counts.get(i, 0)] for i in range(3)],
        )
        detail.value += f'Vorhergesagt: {ZUEGE[rec["prognose"]]}. ' + (
            "Keine Daten: zufällig geraten."
            if total == 0
            else "Häufigste Fortsetzung; Gleichstände zufällig entschieden."
        )

    for i, b in enumerate(buttons):
        b.on_click(lambda _, g=i: play(g))
    reset.on_click(new)
    order.observe(new, names="value")
    display(W.VBox([order, W.HBox(buttons), reset, score, last, box]))
    new()
    PANELS["spiel"] = r
    return r


def station_tabelle():
    art = W.Dropdown(options=GEGNER, value="Viererfolge")
    order = W.Dropdown(options=[1, 2, 3], value=2, description="Ordnung:")
    n = regler(20, 200, 1)
    state = W.Dropdown(layout=W.Layout(width="420px", max_width="100%"))
    info, seq = W.HTML(), W.HTML()
    btn = knopf()
    r = {
        "art": art,
        "order": order,
        "n": n,
        "state": state,
        "button": btn,
        "seed": 2026,
        "last": None,
        "lock": False,
    }

    def show(_=None):
        if r["lock"]:
            return
        counts = r["table"].get(state.value, {})
        total = sum(counts.values())
        info.value = tabelle(
            ["Folgezug", "Anzahl", "Geschätzte Wahrscheinlichkeit"],
            [
                [
                    ZUEGE[i],
                    counts.get(i, 0),
                    (
                        f"{counts.get(i,0)}/{total} = {counts.get(i,0)/total:.3f}"
                        if total
                        else "nicht schätzbar"
                    ),
                ]
                for i in range(3)
            ],
        )
        if not total:
            info.value += "Für diesen Kontext fehlen Daten. Der Computer würde zufällig raten."
        r["last"] = {"counts": dict(counts), "total": total, "history": r["history"].copy()}

    def update(_=None):
        from itertools import product

        r["lock"] = True
        x = gegnerfolge(art.value, n.value, r["seed"])
        table = defaultdict(Counter)
        for t in range(order.value, len(x)):
            table[tuple(x[t - order.value : t])][int(x[t])] += 1
        contexts = list(product(range(3), repeat=order.value))
        old = state.value
        state.options = [(kontextname(k), k) for k in contexts]
        state.value = (
            old
            if old in contexts
            else (max(table, key=lambda k: sum(table[k].values())) if table else contexts[0])
        )
        r["table"] = table
        r["history"] = x
        seq.value = (
            "Beobachtete Folge (erste höchstens 24 Züge): "
            + kontextname(x[:24])
            + (" …" if len(x) > 24 else "")
        )
        r["lock"] = False
        show()

    def new(_):
        r["seed"] += 1
        update()

    for w in (art, order, n):
        w.observe(update, names="value")
    state.observe(show, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                art,
                order,
                feld("Anzahl bereits beobachteter Züge", n),
                btn,
                seq,
                feld("Letzte Züge: Kontext für den nächsten Zug", state),
                info,
            ]
        )
    )
    update()
    PANELS["tabelle"] = r
    return r


def station_vergleich(ordnungen=False):
    art = W.Dropdown(options=GEGNER, value="Viererfolge" if ordnungen else "Dreierzyklus")
    n = regler(300, 3000, 10, 10)
    repeat = regler(80, 100, 0)
    btn = knopf()
    pic = bild()
    info = W.HTML()
    labels = (
        ["Ordnung 1", "Ordnung 2", "Ordnung 3"]
        if ordnungen
        else ["Lernender Computer (Ordnung 2)", "Zufälliger Computer"]
    )
    fig, ax = plt.subplots(figsize=(6.7, 3.6))
    lines = [
        ax.plot([], [], label=s, ls=("--" if ordnungen and i == 2 else "-"))[0]
        for i, s in enumerate(labels)
    ]
    ax.axhline(1 / 3, color="grey", ls=":", label="1/3")
    ax.set(
        xscale="log",
        xlim=(1, 300),
        ylim=(-0.03, 1.05),
        xlabel="Runden n (logarithmisch)",
        ylabel="Bisheriger Siegesanteil\ndes Computers",
        yticks=np.linspace(0, 1, 6),
    )
    ax.legend(fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.02), ncol=2, frameon=False)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.close(fig)
    r = {
        "art": art,
        "n": n,
        "repeat": repeat,
        "button": btn,
        "seed": 2026,
        "key": None,
        "bild": pic,
        "last": None,
    }

    def update(_=None):
        repeat.disabled = art.value != "Häufig wiederholen"
        key = (art.value, repeat.value if not repeat.disabled else None, r["seed"])
        if r["key"] != key:
            r["opponent"] = gegnerfolge(art.value, 3000, r["seed"], repeat.value / 100)
            r["runs"] = [
                simuliere(r["opponent"], o, r["seed"])
                for o in ([1, 2, 3] if ordnungen else [2])
            ]
            if not ordnungen:
                r["runs"].append({"scores": zufalls_computer(r["opponent"], r["seed"])})
            r["key"] = key
        N = n.value
        idx = np.unique(np.geomspace(1, N, min(N, 600)).astype(int))
        rows = []
        for label, line, run in zip(labels, lines, r["runs"]):
            scores = run["scores"][:N]
            line.set_data(idx, siegkurve(scores)[idx - 1])
            row = [
                label,
                int((scores == 1).sum()),
                int((scores == -1).sum()),
                int((scores == 0).sum()),
            ]
            if ordnungen:
                row.append(f'{run["contexts"][N-1]} / {3**(len(rows)+1)}')
            rows.append(row)
        ax.set_xlim(1, N)
        zeichnen(fig, pic)
        info.value = tabelle(
            ["Computer", "Siege", "Niederlagen", "Unentschieden"]
            + (["Beobachtete / mögliche Kontexte"] if ordnungen else []),
            rows,
        )
        r["last"] = {
            "opponent": r["opponent"][:N].copy(),
            "scores": [run["scores"][:N].copy() for run in r["runs"]],
        }

    def new(_):
        r["seed"] += 1
        update()

    for w in (art, n, repeat):
        w.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                art,
                feld("Wiederholwahrscheinlichkeit in % (nur beim Wiederholgegner)", repeat),
                feld("Ausgewertete Runden", n),
                btn,
                pic,
                info,
            ]
        )
    )
    update()
    PANELS["ordnungen" if ordnungen else "vergleich"] = r
    return r


def fenstergegner(n, seed, wechsel, p=0.45):
    """Markov-Kette: +1 mit p, sonst 0/-1 gleich häufig; ab Mitte ggf. -1 bevorzugt."""
    rng = random.Random(seed)
    x = np.empty(n, dtype=int)
    x[0] = rng.randrange(3)
    for t in range(1, n):
        schritt = 2 if wechsel and t >= n // 2 else 1
        x[t] = (
            x[t - 1]
            + (
                schritt
                if rng.random() < p
                else rng.choice([i for i in range(3) if i != schritt])
            )
        ) % 3
    return x


def rollende_siege(scores, breite=50):
    end = np.arange(1, scores.shape[-1] + 1)
    begin = np.maximum(0, end - breite)
    sums = np.concatenate(
        [np.zeros((*scores.shape[:-1], 1)), np.cumsum(scores == 1, axis=-1)], axis=-1
    )
    return (sums[..., end] - sums[..., begin]) / (end - begin)


def fenstervergleich(Wert, seed=2026, wiederholungen=64, n=600):
    scores = np.empty((2, 2, wiederholungen, n), dtype=np.int8)
    folgen = np.empty((2, wiederholungen, n), dtype=np.int8)
    for fall in range(2):
        for r in range(wiederholungen):
            folge = fenstergegner(n, seed + 100003 * r, wechsel=(fall == 1))
            folgen[fall, r] = folge
            for j, w in enumerate((None, Wert)):
                scores[fall, j, r] = simuliere(folge, 1, seed + 200003 * r + 99991, fenster=w)[
                    "scores"
                ]
    return {"scores": scores, "opponent": folgen}


def station_wechsel():
    window = W.SelectionSlider(
        options=[3, 9, 30, 60, 90, 150, 300, 600],
        value=9,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )
    btn = knopf("64 neue Versuche")
    pic = bild()
    pic.layout.width = "950px"
    info = W.HTML()
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.7), sharey=True)
    lines = []
    for ax, title in zip(
        axes, ["Spielstrategie 1", "Spielstrategie 2 (mit Wechsel nach Runde 300)"]
    ):
        lines.append(
            [
                ax.plot([], [], label=l, ls=st)[0]
                for l, st in [("Alle Übergänge", "-"), ("Letzte W Übergänge", "--")]
            ]
        )
        ax.set_title(title, fontsize=10)
        ax.set(
            xlim=(1, 600),
            ylim=(0.20, 0.55),
            yticks=np.arange(0.20, 0.551, 0.05),
            xlabel="Runde",
        )
        ax.grid(alpha=0.2)
        ax.axhline(0.45, color="grey", ls=":", lw=1)
    axes[1].axvline(300.5, color="grey", ls=":")
    axes[0].set_ylabel("Siegesanteil in letzten bis zu 50 Runden\n(Mittel über 64 Versuche)")
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    plt.close(fig)
    r = {"window": window, "button": btn, "seed": 2026, "bild": pic, "last": None, "cache": {}}

    def update(_=None):
        key = (window.value, r["seed"])
        if key not in r["cache"]:
            r["cache"][key] = fenstervergleich(window.value, r["seed"])
        data = r["cache"][key]
        scores = data["scores"]
        curves = rollende_siege(scores).mean(axis=2)
        for f in range(2):
            for j in range(2):
                lines[f][j].set_data(np.arange(1, 601), curves[f, j])
        zeichnen(fig, pic)
        rates = (scores[:, :, :, 300:] == 1).mean(axis=(2, 3))
        info.value = tabelle(
            [
                "Siegesanteil in Runden 301–600",
                "Alle Übergänge",
                f"Letzte {window.value} Übergänge",
            ],
            [
                [name, f"{a:.1%}", f"{b:.1%}"]
                for name, (a, b) in zip(
                    ["Spielstrategie 1", "Spielstrategie 2 (mit Wechsel)"], rates
                )
            ],
        )
        info.value += "Beide Computer erhalten je Versuch dieselben Gegnerzüge. Gepunktete Horizontale: 45 % bei bekannter Gegnerregel. Beide y-Achsen zeigen 20–55 %. W = 600 entspricht allen Übergängen."
        r["last"] = {**data, "curves": curves, "rates": rates}

    def new(_):
        r["seed"] += 10000019
        r["cache"].clear()
        update()

    window.observe(update, names="value")
    btn.on_click(new)
    display(W.VBox([feld("W: so viele letzte Übergänge behalten wir", window), btn, pic, info]))
    update()
    PANELS["wechsel"] = r
    return r


STRATEGIEN = (
    "Ordnung 1",
    "Ordnung 2",
    "Zugpaar",
    "Ergebnis → Wechsel",
    "Letzten Zug kontern",
    "Bleiben/Wechseln",
    "Zufall",
)
STRATEGIE_INFO = {
    "Ordnung 1": "Lernt den nächsten Gegnerzug aus dessen letztem Zug (3 Zustände).",
    "Ordnung 2": "Lernt den nächsten Gegnerzug aus dessen letzten zwei Zügen (9 Zustände).",
    "Zugpaar": "Lernt den nächsten Gegnerzug aus dem Paar (eigener letzter Zug, letzter Gegnerzug).",
    "Ergebnis → Wechsel": "Lernt nach eigenem Sieg, Niederlage oder Unentschieden, wie der Gegner seinen Zug verändert: gleich, +1 oder +2 im Kreis Schere → Stein → Papier → Schere",
    "Letzten Zug kontern": "Kontert den letzten Zug des anderen Spielers; in Runde 1 zufällig. Eine fixe Strategie, die nicht lernt.",
    "Bleiben/Wechseln": "Behält nach eigenem Sieg den Zug bei; nach Niederlage oder Unentschieden geht es einen Schritt im Kreis weiter. Eine fixe Strategie, die nicht lernt.",
    "Zufall": "Wählt in jeder Runde unabhängig gleichverteilt.",
}
LERNENDE = STRATEGIEN[:4]


# Spielstrategien und Turnier.
class DuellSpieler:
    """Zugwahl und Lernen getrennt: beide wählen, bevor einer die neuen Züge sieht."""

    def __init__(self, art, seed, erkundung=0.0):
        if art not in STRATEGIEN:
            raise ValueError("Unbekannte Strategie")
        if not 0 <= erkundung <= 1:
            raise ValueError("Erkundung muss zwischen 0 und 1 liegen.")
        self.erkundung = erkundung
        self.art = art
        self.rng = random.Random(seed)
        self.eigene = []
        self.fremde = []
        self.tabelle = defaultdict(Counter)

    def kontext(self):
        if not self.fremde:
            return None
        if self.art == "Ordnung 1":
            return (self.fremde[-1],)
        if self.art == "Ordnung 2":
            return tuple(self.fremde[-2:]) if len(self.fremde) >= 2 else None
        if self.art == "Zugpaar":
            return (self.eigene[-1], self.fremde[-1])
        if self.art == "Ergebnis → Wechsel":
            return (ergebnis(self.eigene[-1], self.fremde[-1]),)
        return None

    def waehle(self):
        if self.art == "Zufall" or not self.fremde:
            return self.rng.randrange(3)
        if self.art == "Letzten Zug kontern":
            return konter(self.fremde[-1])
        if self.art == "Bleiben/Wechseln":
            return (
                self.eigene[-1]
                if ergebnis(self.eigene[-1], self.fremde[-1]) == 1
                else konter(self.eigene[-1])
            )
        # Zusätzliche Zufallszüge nur für Lernende; bei 0 % bleiben zufällige Gleichstände.
        if self.rng.random() < self.erkundung:
            return self.rng.randrange(3)
        counts = self.tabelle.get(self.kontext(), {})
        m = max(counts.values(), default=0)
        ziel = self.rng.choice([i for i in range(3) if counts.get(i, 0) == m])
        prognose = (self.fremde[-1] + ziel) % 3 if self.art == "Ergebnis → Wechsel" else ziel
        return konter(prognose)

    def lerne(self, eigener, fremder):
        k = self.kontext()
        if self.art in LERNENDE and k is not None:
            ziel = (
                (fremder - self.fremde[-1]) % 3 if self.art == "Ergebnis → Wechsel" else fremder
            )
            self.tabelle[k][ziel] += 1
        self.eigene.append(eigener)
        self.fremde.append(fremder)


def duell(a, b, n, seed_a, seed_b, erkundung=0.0):
    A = DuellSpieler(a, seed_a, erkundung)
    B = DuellSpieler(b, seed_b, erkundung)
    moves = np.empty((n, 2), dtype=np.int8)
    scores = np.empty(n, dtype=np.int8)
    for t in range(n):
        x = A.waehle()
        y = B.waehle()  # Beide Entscheidungen ohne den aktuellen fremden Zug.
        moves[t] = (x, y)
        scores[t] = ergebnis(x, y)
        A.lerne(x, y)
        B.lerne(y, x)
    return {"scores": scores, "moves": moves, "A": A, "B": B}


def turnier(seed=2026, n=1000, wiederholungen=6, erkundung=0.0):
    # Jede Paarung beginnt sechsmal ohne Vorwissen; getrennte Zufallsquellen.
    # Je zwei Versuche vertauschen die Zuordnung der Zufallsquellen zu A und B.
    scores = np.zeros((len(STRATEGIEN), len(STRATEGIEN), wiederholungen, n), dtype=np.int8)
    for i, a in enumerate(STRATEGIEN):
        for j in range(i + 1, len(STRATEGIEN)):
            for r in range(wiederholungen):
                sa = seed + 1000003 * (i * len(STRATEGIEN) + j) + 10007 * (r // 2)
                sb = sa + 500009
                if r % 2:
                    sa, sb = sb, sa
                s = duell(a, STRATEGIEN[j], n, sa, sb, erkundung)["scores"]
                scores[i, j, r] = s
                scores[j, i, r] = -s
    return scores


def station_strategien():
    n = W.SelectionSlider(
        options=[50, 100, 300, 1000],
        value=300,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )
    A = W.Dropdown(
        options=STRATEGIEN,
        value="Zugpaar",
        description="Spieler A:",
        layout=W.Layout(width="310px"),
    )
    B = W.Dropdown(
        options=STRATEGIEN,
        value="Letzten Zug kontern",
        description="Spieler B:",
        layout=W.Layout(width="310px"),
    )
    btn = knopf("Neues Turnier")
    pic = bild()
    pic.layout.width = "850px"
    detail = bild()
    info = W.HTML()
    regeln = W.HTML()
    fig, ax = plt.subplots(figsize=(8, 5.2))
    mask = np.eye(len(STRATEGIEN), dtype=bool)
    cmap = plt.get_cmap("RdBu").copy()
    cmap.set_bad("#eeeeee")
    im = ax.imshow(np.ma.array(np.zeros(mask.shape), mask=mask), vmin=-1, vmax=1, cmap=cmap)
    ax.set_xticks(range(len(STRATEGIEN)), STRATEGIEN, rotation=30, ha="right", fontsize=9)
    ax.set_yticks(range(len(STRATEGIEN)), STRATEGIEN, fontsize=9)
    ax.set(xlabel="Gegner (Spalte)", ylabel="Strategie (Zeile)")
    texts = [
        [
            ax.text(j, i, "—" if i == j else "", ha="center", va="center", fontsize=9)
            for j in range(len(STRATEGIEN))
        ]
        for i in range(len(STRATEGIEN))
    ]
    fig.colorbar(im, ax=ax, label="(Siege − Niederlagen) / Runden", fraction=0.05, pad=0.04)
    fig.tight_layout()
    plt.close(fig)
    f2, ax2 = plt.subplots(figsize=(6.7, 3.4))
    line = ax2.plot([], [], color="#17699e")[0]
    ax2.axhline(0, color="grey", ls=":")
    ax2.set(
        ylim=(-1, 1),
        xlabel="Runden",
        ylabel="Bisheriger Punktesaldo pro Runde\naus Sicht von A",
    )
    ax2.grid(alpha=0.2)
    f2.tight_layout()
    plt.close(f2)
    r = {
        "n": n,
        "A": A,
        "B": B,
        "button": btn,
        "seed": 2026,
        "bild": pic,
        "detail": detail,
        "last": None,
        "scores": None,
        "cache": {},
    }

    def auswahl(_=None):
        i = STRATEGIEN.index(A.value)
        j = STRATEGIEN.index(B.value)
        N = n.value
        regeln.value = (
            "<b>A:</b> " + STRATEGIE_INFO[A.value] + "<br><b>B:</b> " + STRATEGIE_INFO[B.value]
        )
        if i == j:
            line.set_data([], [])
            zeichnen(f2, detail)
            info.value = (
                "Wähle zwei verschiedene Strategien. Die graue Diagonale wurde nicht simuliert."
            )
            r["selected"] = None
            return
        s = r["scores"][i, j, :, :N]
        curve = s.cumsum(axis=1).mean(axis=0) / np.arange(1, N + 1)
        line.set_data(np.arange(1, N + 1), curve)
        ax2.set_xlim(1, N)
        zeichnen(f2, detail)
        rates = np.array([(s == v).mean() for v in (1, 0, -1)])
        info.value = tabelle(
            ["A gewinnt", "Unentschieden", "B gewinnt", "Punktesaldo pro Runde für A"],
            [[f"{v:.1%}" for v in rates] + [f"{s.mean():+.3f}"]],
        )
        info.value += "Mittel über sechs Spiele."
        r["selected"] = {"scores": s.copy(), "curve": curve, "rates": rates}

    def update(_=None):
        key = r["seed"]
        if key not in r["cache"]:
            r["cache"][key] = turnier(r["seed"], erkundung=0.0)
        r["scores"] = r["cache"][key]
        matrix = r["scores"][:, :, :, : n.value].mean(axis=(2, 3))
        im.set_data(np.ma.array(matrix, mask=mask))
        for i in range(len(STRATEGIEN)):
            for j in range(len(STRATEGIEN)):
                if i != j:
                    texts[i][j].set_text(f"{matrix[i,j]:+.2f}")
                    texts[i][j].set_color("white" if abs(matrix[i, j]) > 0.6 else "black")
        zeichnen(fig, pic)
        r["last"] = {"matrix": matrix}
        auswahl()

    def new(_):
        r["seed"] += 100000007
        r["cache"].clear()
        update()

    n.observe(update, names="value")
    A.observe(auswahl, names="value")
    B.observe(auswahl, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("Ausgewertete Runden je Spiel", n),
                btn,
                W.HTML(
                    "Blau: Vorteil der Zeilenstrategie · Rot: Nachteil · Grau: nicht simuliert"
                ),
                pic,
                W.HBox([A, B]),
                regeln,
                detail,
                info,
            ]
        )
    )
    update()
    PANELS["strategien"] = r
    return r


print("Bereit – weiter mit Station 1.")


Bereit – weiter mit Station 1.


<br><br>

## 1 · Einen Zug vorhersagen

Wir haben diese Folge beobachtet:

> Schere · Stein · Papier · Schere · Stein · Schere · Stein · Papier · Stein · Stein · Stein · Stein · Schere · Stein

1. **Zähle, welche Züge unmittelbar nach dem Paar „Schere, Stein“ vorkamen.** Welche Wahrscheinlichkeiten schätzt du daraus für Schere, Stein und Papier? <span style="color: #707070;">(leicht)</span>
2. Welchen Zug sollte der Computer wählen, um die häufigste Fortsetzung des letzten Zustands "Schere, Stein" zu schlagen? <span style="color: #707070;">(leicht)</span>
3. Ein **Modell zweiter Ordnung** verwendet die letzten beiden Gegnerzüge als Kontext, ein Modell erster Ordnung verwendet nur den letzten. Welchen Zug sollte der Computer nach "Stein" wählen, wenn er statt eines Modells zweiter Ordnung ein Modell erster Ordnung verwendet? <span style="color: #707070;">(schwerer)</span>


In [2]:
station_1 = station_vorhersage()

<br><br>

## 2 · Spiele selbst

Wähle deinen Zug über einen der drei Knöpfe. Nach jeder Runde wird dein neuer Zug in die Häufigkeitstabelle aufgenommen. Daraus leitet der Computer seine Prognose ab.

1. **Spiele zunächst etwa 20 Runden nach einem wiederkehrenden Muster, z.B. Schere → Stein → Papier → Schere → ....** Ab wann gewinnt der Computer häufiger? Beginne dann ein neues Spiel und versuche, unvorhersehbar zu spielen. <span style="color: #707070;">(leicht)</span>
2. **Versuche, in insgesamt 50 Runden den Computer häufiger zu schlagen als er dich.** Probiere sowohl Ordnung 1, 2 als auch 3 für das Computermodell aus. <span style="color: #707070;">(leicht)</span>


In [3]:
station_2 = station_spielen()

<br><br>

## 3 · Was steckt in der gelernten Tabelle?

Hier betrachten wir die folgenden Spielmuster des menschlichen Spielers:

- **Dreierzyklus**: Schere → Stein → Papier → Schere → Stein → Papier → ...
- **Viererfolge**: Schere → Schere → Stein → Papier → Schere → Schere → ...
- **Häufig wiederholen**: Mit Wahrscheinlichkeit 80% wird der letzte Zug wiederholt, mit Wahrscheinlichkeit 20% wird gleichverteilt einer der beiden anderen Züge gewählt.
- **Gleichverteilung**: Unabhängig gleichverteilte Züge in jedem Schritt.

1. **Wähle "Viererfolge" und beginne mit Ordnung 1 und dem Zustand Schere.** Warum gibt es zwei mögliche Nachfolger? Untersuche danach bei Ordnung 2 die Zustände „Papier, Schere“ und „Schere, Schere“. <span style="color: #707070;">(schwerer)</span>
2. **Wähle "Gleichverteilung", Ordnung 3 und den Zustand "Papier, Schere, Papier"**  und verändere die Zahl beobachteter Züge mithilfe des Reglers. Warum ist ein Modell höherer Ordnung bei vorgegebener Maximalanzahl an Zügen nicht unbedingt "besser" als ein Modell niedrigerer Ordnung? Zähle Vor- und Nachteile auf. <span style="color: #707070;">(schwerer)</span>


In [4]:
station_3 = station_tabelle()

<br><br>

## 4 · Gegen welche Muster hilft Lernen?

Vergleiche den lernenden Computer **mit Ordnung 2** mit einem Computer, der unabhängig gleichverteilt spielt. Beide erhalten dieselbe Gegnerfolge. Die Kurven zeigen ihre bisherigen **Siegesanteile**, die Tabelle zusätzlich Niederlagen und Unentschieden. Für die Spielstrategie "Häufig wiederholen" kannst du außerdem die Wiederholwahrscheinlichkeit $p$ in % einstellen (In Station 3 war $p = 80$%).

1. Vermute zuerst, gegen welche der vier Spielstrategien (Dreierzyklus, Viererfolge, Häufig wiederholen, Gleichverteilung) Lernen hilft und gegen welche nicht. <span style="color: #707070;">(leicht)</span>
2. Warum konvergiert der Siegesanteil des unabhängig gleichverteilt spielenden Computers immer gegen 1/3, egal gegen welche der vier Spielstrategien er antritt? <span style="color: #707070;">(schwerer)</span>
3. Begründe, warum der Spiegesanteil des lernenden Computers gegen die Spielstrategie "Häufig wiederholen" mit Wiederholwahrscheinlichkeit $p$ gegen $p$ konvergiert, falls $p \geq \frac{1}{3}$, aber gegen $\frac{1-p}{2}$ konvergiert, falls $p \leq \frac{1}{3}$. <span style="color: #707070;">(schwer)</span>


In [5]:
station_4 = station_vergleich(ordnungen=False)

<br><br>

## 5 · Hilft eine höhere Ordnung immer?

Nun vergleichen wir die Ordnungen **1, 2 und 3**, wenn sie jeweils gegen die selbe Spielstrategie antreten (Dreierzyklus, Viererfolge, Häufig wiederholen, Gleichverteilung).

1. **Untersuche zunächst die Viererfolge.** Warum führt ein Computermodell der Ordnung 2 oder 3 hier langfristig zu einer Siegchance von 100%, aber nicht ein Computermodell der Ordnung 1? <span style="color: #707070;">(schwerer)</span>
2. Wogegen konvergiert der langfristige Sieganteil des Computermodells der Ordnung 1 gegen die Viererfolge? <span style="color: #707070;">(schwerer)</span>
3. Erkläre am Beispiel der Spielstrategie "Häufig wiederholen", warum eine höhere Ordnung des Computermodells bei wenigen Daten auch nachteilig sein kann. <span style="color: #707070;">(schwerer)</span>


In [6]:
station_5 = station_vergleich(ordnungen=True)

<br><br>

## 6 · Wie viel Vergangenheit hilft?

Wir lassen den lernenden Computer gegen zwei Spielstrategien antreten:

- **Spielstrategie 1**: Mit Wahrscheinlichkeit **45 %** wird im Kreis Schere → Stein → Papier → Schere → … weitergespielt; die beiden anderen Möglichkeiten haben jeweils **27,5 %** Wahrscheinlichkeit.
- **Spielstrategie 2**: Wie Spielstrategie 1, aber ab Runde 301 wird die bevorzugte Richtung umgedreht: Schere → Papier → Stein → Schere → … . Die Wahrscheinlichkeiten bleiben 45 %, 27,5 %, 27,5 %.

Der Computer verwendet jeweils **Ordnung 1**: einmal alle bisherigen Übergänge (blau), einmal nur die letzten **W Übergänge** (orange).

1. Beginne mit **W = 9** und vergrößere das Fenster. Welche Einstellung erzielt bei Spielstrategie 2 in den Runden 301–600 den höchsten Siegesanteil? Nutze die Tabelle und prüfe mit neuen Versuchen, ob sich eine einzelne Fenstergröße eindeutig abhebt. <span style="color: #707070;">(leicht)</span>
2. Vergleiche diese Einstellung mit dem Lernen aus allen Übergängen bei **Spielstrategie 1**. Warum hilft mehr Vergangenheit dort, kann aber bei Spielstrategie 2 schaden? <span style="color: #707070;">(schwerer)</span>

Die Kurven zeigen die Siegesanteile in den letzten bis zu 50 Runden, gemittelt über **64 Versuche**. Beide y-Achsen zeigen denselben Ausschnitt von **20 % bis 55 %**. Die gepunktete Horizontale markiert die Siegchance von 45 %, wenn die Gegnerregel bekannt ist.


In [7]:
station_6 = station_wechsel()

<br><br>

## 7 · Welche Strategie schlägt welche?

Wir wollen nun verschiedene Spielstrategien des Computers betrachten. Die ersten zwei Strategien sind bereits bekannt:

- **Ordnung 1**: Lernt den nächsten Gegnerzug aus dessen letztem Zug (3 Zustände).
- **Ordnung 2**: Lernt den nächsten Gegnerzug aus dessen letzten zwei Zügen (9 Zustände).

Hinzu kommen nun folgende Spielstrategien des Computers:

- **Zugpaar**: Lernt den nächsten Gegnerzug aus dem Paar (letzter eigener Zug, letzter Gegnerzug).
- **Ergebnis → Wechsel**: Lernt nach einem eigenem Sieg, einer eigenen Niederlage oder einem Unentschieden, wie der Gegner seinen letzten Zug verändert: gar nicht, +1 oder +2 im Kreis Schere → Stein → Papier → Schere → ...

Außerdem betrachten wir folgende fixe Spielstrategien, die gar nicht lernen:
- **Letzten Zug kontern**: Kontert den letzten Zug des anderen Spielers.
- **Bleiben/Wechsel**: Behält nach eigenem Sieg den letzten Zug bei; nach Niederlage oder Unentschieden geht es einen Schritt im Kreis Schere → Stein → Papier → Schere → ... weiter
- **Zufall**: Wählt in jeder Runde unabhängig gleichverteilt.

Wir lassen nun je zwei der obigen Computerstrategien in einer Simulation für zunächst 300 Runden gegeneinander antreten und notieren uns danach den durchschnittlichen Punktesaldo pro Runde einer jeden Strategie (Gewinnen zählt 1 Punkt, Unentschieden zählt 0 Punkte, Verlieren zählt -1 Punkt). Die Matrix unten beinhaltet all diese durchschnittlichen Punktesaldos.

1. Vergleiche **Ordnung 1** und **Ordnung 2** jeweils gegen **Letzten Zug kontern**. Welche erzielt den höheren Punktesaldo? Prüfe auch mit neuen Versuchen. <span style="color: #707070;">(leicht)</span>
2. Warum ist "Ordnung 2" so viel erfolgreicher als "Ordnung 1" gegen "Letzten Zug kontern"? <span style="color: #707070;">(schwerer)</span>
3. Warum ist **Zugpaar** so erfolgreich gegen **Letzten Zug kontern**? <span style="color: #707070;">(schwerer)</span>
4. Warum ist **Ergebnis → Wechsel** so erfolgreich gegen **Bleiben/Wechseln**? <span style="color: #707070;">(schwerer)</span>
5. Wie viele verschiedene Kontexte verwenden die neuen lernenden Strategien **Zugpaar** und **Ergebnis → Wechsel** für ihre Vorhersage? Beschreibe auch, welche vergangenen Züge bzw. Ergebnisse die festen Strategien für ihren nächsten Zug benötigen. Gemeint ist die Information für die nächste Entscheidung; die gelernten Häufigkeitstabellen kommen beim lernenden Computer noch hinzu. <span style="color: #707070;">(schwerer)</span>
6. Kann man eine Strategie konstruieren, die systematisch erfolgreich gegen **Ordnung 1** ist? <span style="color: #707070;">(schwer)</span>
7. Kann man eine Strategie konstruieren, die systematisch erfolgreich gegen **jede** Ordnung ist, ohne die Ordnung des Computergegners vorher zu kennen? <span style="color: #707070;">(schwer)</span>

**Blau bedeutet Vorteil der Zeile gegenüber der Spalte.** Gemessen wird $\text{Punktesaldo} = (\text{Siege}-\text{Niederlagen})/\text{Runden}$. Die lernenden Strategien kontern stets ihre Prognose.


In [8]:
station_7 = station_strategien()

<br><br>

## Kleine Codeaufgabe · Den Konter selbst wählen (optional) <span style="color: #707070;">(leicht)</span>

Das Wörterbuch `antwort` ordnet jedem vorhergesagten Gegnerzug den schlagenden Computerzug zu. **Ersetze `None` durch den passenden Zugriff auf dieses Wörterbuch.** Allgemein liest `woerterbuch[schluessel]` den zugehörigen Wert aus.

Probiere anschließend alle drei Prognosen. Warum sagt die korrekte Zuordnung noch nicht, ob der Computer die tatsächliche Runde gewinnt?

In [9]:
antwort = {"Schere": "Stein", "Stein": "Papier", "Papier": "Schere"}
prognose = "Papier"
computerzug = None  # Ergänze hier den Zugriff auf antwort.

if computerzug is None:
    print("Ersetze None durch den passenden Wörterbuchzugriff.")
else:
    print("Vorhergesagter Gegnerzug:", prognose)
    print("Dein gewählter Computerzug:", computerzug)


Ersetze None durch den passenden Wörterbuchzugriff.
